In [5]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/Mobility_Hackathon/preprocessed/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_metadata.pkl  val_metadata.pkl


In [7]:
"""
Single-image BEV segmentation training + testing pipeline for your hackathon dataset.

Why this version is different from the earlier one:
- Your PKL samples have keys like: image_path, camera, sample_token, scene_token, K_scaled, mask, is_train, drivable_fraction.
- That means this is NOT a 6-camera LSS dataset.
- You also hit a sympy/timm import issue, so this version removes timm completely.
- This model is a direct image -> BEV mask network that matches your actual data format.

Install in Colab if needed:
    !pip -q install pillow

Typical Colab usage:
    from google.colab import drive
    drive.mount('/content/drive')

    cfg = CFG()
    best_ckpt = run_training(cfg)
    run_testing(cfg, best_ckpt)
"""

import os
import math
import time
import pickle
import json
import random
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

@dataclass
class CFG:
    train_pkl: Optional[str] = None
    val_pkl: Optional[str] = None
    out_dir: str = "/content/drive/MyDrive/Mobility Hackathon/checkpoints"

    # Input image size
    image_h: int = 256
    image_w: int = 448

    # If None, the dataset will infer mask size from the first sample.
    mask_h: Optional[int] = None
    mask_w: Optional[int] = None

    batch_size: int = 4
    num_workers: int = 2
    epochs: int = 25
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    amp: bool = True
    accum_steps: int = 2
    seed: int = 42
    threshold: float = 0.5
    save_name: str = "best_singleview_bev.pt"


# -----------------------------------------------------------------------------
# Helper utilities
# -----------------------------------------------------------------------------

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def resolve_drive_file(filename: str, preferred_paths: Optional[Sequence[str]] = None) -> str:
    """Find a file in Drive without manually editing the notebook every time."""
    candidates = list(preferred_paths or [])
    candidates += [
        f"/content/drive/MyDrive/{filename}",
        f"/content/drive/MyDrive/Mobility Hackathon/preprocessed/{filename}",
        f"/content/drive/MyDrive/Mobility_Hackathon/preprocessed/{filename}",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    root = "/content/drive/MyDrive"
    if os.path.exists(root):
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                return os.path.join(dirpath, filename)

    raise FileNotFoundError(f"Could not find {filename} under {root}")


def load_pickle(path: str) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)


def unwrap_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ["samples", "data", "records", "items", "metas", "metadata"]:
            if key in obj and isinstance(obj[key], list):
                return obj[key]
    raise TypeError(f"Unsupported pickle structure: {type(obj)}")


def inspect_pkl(path: str, n: int = 1) -> None:
    data = unwrap_records(load_pickle(path))
    print(f"Loaded {len(data)} records from: {path}")
    for i in range(min(n, len(data))):
        print(f"\nSample {i} keys:")
        print(list(data[i].keys()))


def _first(d: Dict[str, Any], keys: Sequence[str]) -> Any:
    for k in keys:
        if k in d:
            return d[k]
    raise KeyError(f"None of these keys were found: {keys}. Available keys: {list(d.keys())[:30]}")


def resolve_path(p: Any, base_dir: str = "") -> Any:
    if not isinstance(p, str):
        return p
    if os.path.isabs(p) and os.path.exists(p):
        return p
    if base_dir:
        candidate = os.path.join(base_dir, p)
        if os.path.exists(candidate):
            return candidate
    if os.path.exists(p):
        return p
    return p


def load_image_any(item: Any, image_size: Tuple[int, int], base_dir: str = "") -> Tuple[torch.Tensor, Tuple[int, int]]:
    """Return CHW float tensor in [0,1] and original size (H,W)."""
    item = resolve_path(item, base_dir=base_dir)

    if isinstance(item, dict):
        item = _first(item, ["path", "img_path", "image_path", "file_path", "filename", "data_path"])
        item = resolve_path(item, base_dir=base_dir)

    if isinstance(item, str):
        img = Image.open(item).convert("RGB")
        orig_w, orig_h = img.size
        img = img.resize((image_size[1], image_size[0]), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        return tensor, (orig_h, orig_w)

    if isinstance(item, np.ndarray):
        arr = item
        if arr.ndim == 2:
            arr = np.repeat(arr[..., None], 3, axis=2)
        if arr.shape[-1] == 4:
            arr = arr[..., :3]
        if np.issubdtype(arr.dtype, np.floating):
            arr = np.clip(arr, 0.0, 1.0)
            if arr.max() <= 1.0:
                arr = (arr * 255.0).astype(np.uint8)
        orig_h, orig_w = arr.shape[:2]
        img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
        img = img.resize((image_size[1], image_size[0]), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        return tensor, (orig_h, orig_w)

    if isinstance(item, torch.Tensor):
        return load_image_any(item.detach().cpu().numpy(), image_size, base_dir=base_dir)

    raise TypeError(f"Unsupported image type: {type(item)}")


def load_mask_any(mask: Any, size: Tuple[int, int], base_dir: str = "") -> torch.Tensor:
    mask = resolve_path(mask, base_dir=base_dir)

    if isinstance(mask, dict):
        mask = _first(mask, ["path", "mask_path", "label_path", "target_path"])
        mask = resolve_path(mask, base_dir=base_dir)

    if isinstance(mask, str):
        m = Image.open(mask).convert("L")
        m = m.resize((size[1], size[0]), Image.NEAREST)
        arr = np.asarray(m, dtype=np.float32)
        if arr.max() > 1.0:
            arr = arr / 255.0
        return torch.from_numpy(arr).unsqueeze(0)

    if isinstance(mask, np.ndarray):
        arr = mask
        if arr.ndim == 3 and arr.shape[0] == 1:
            arr = arr[0]
        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = arr[..., 0]
        if arr.ndim != 2:
            raise ValueError(f"Mask must be 2D, got shape {arr.shape}")
        if arr.max() > 1.0:
            arr = arr / 255.0
        m = Image.fromarray((arr * 255).astype(np.uint8)).convert("L")
        m = m.resize((size[1], size[0]), Image.NEAREST)
        arr = np.asarray(m, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).unsqueeze(0)

    if isinstance(mask, torch.Tensor):
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        if mask.ndim == 3 and mask.shape[0] != 1:
            mask = mask[:1]
        mask = mask.float()
        if mask.max() > 1.0:
            mask = mask / 255.0
        mask = F.interpolate(mask.unsqueeze(0), size=size, mode="nearest").squeeze(0)
        return mask

    raise TypeError(f"Unsupported mask type: {type(mask)}")


# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------

class SingleViewBEVDataset(Dataset):
    """Dataset for samples that contain one image_path and one BEV mask."""

    def __init__(self, pkl_path: str, image_size: Tuple[int, int], mask_size: Optional[Tuple[int, int]] = None, augment: bool = False):
        super().__init__()
        self.pkl_path = pkl_path
        self.base_dir = os.path.dirname(pkl_path)
        self.records = unwrap_records(load_pickle(pkl_path))
        self.image_size = image_size
        self.augment = augment

        if mask_size is None:
            mask_size = self._infer_mask_size(self.records[0])
        self.mask_size = mask_size

    def _infer_mask_size(self, sample: Dict[str, Any]) -> Tuple[int, int]:
        m = _first(sample, ["mask", "bev_mask", "target", "drivable_mask", "gt_mask", "label"])
        m = resolve_path(m, base_dir=self.base_dir)
        if isinstance(m, str) and os.path.exists(m):
            img = Image.open(m)
            return (img.size[1], img.size[0])
        if isinstance(m, np.ndarray):
            if m.ndim == 2:
                return (int(m.shape[0]), int(m.shape[1]))
            if m.ndim == 3:
                return (int(m.shape[-2]), int(m.shape[-1]))
        if isinstance(m, torch.Tensor):
            if m.ndim == 2:
                return (int(m.shape[0]), int(m.shape[1]))
            if m.ndim == 3:
                return (int(m.shape[-2]), int(m.shape[-1]))
        return (256, 256)

    def __len__(self) -> int:
        return len(self.records)

    def _get_image_path(self, sample: Dict[str, Any]) -> Any:
        return _first(sample, ["image_path", "img_path", "path", "image", "camera_image"])

    def _get_mask(self, sample: Dict[str, Any]) -> Any:
        return _first(sample, ["mask", "bev_mask", "target", "drivable_mask", "gt_mask", "label"])

    def _horizontal_flip(self, img: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        return img, mask

    def _color_jitter(self, img: torch.Tensor) -> torch.Tensor:
        # Small, safe augmentation without extra dependencies.
        if random.random() < 0.8:
            brightness = 1.0 + random.uniform(-0.15, 0.15)
            contrast = 1.0 + random.uniform(-0.15, 0.15)
            mean = img.mean(dim=(1, 2), keepdim=True)
            img = (img - mean) * contrast + mean
            img = img * brightness
            img = img.clamp(0.0, 1.0)
        return img

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.records[idx]
        img_path = self._get_image_path(sample)
        mask_raw = self._get_mask(sample)

        img, _ = load_image_any(img_path, self.image_size, base_dir=self.base_dir)
        mask = load_mask_any(mask_raw, self.image_size, base_dir=self.base_dir)
        mask = (mask > 0.5).float()

        if self.augment:
            img, mask = self._horizontal_flip(img, mask)
            img = self._color_jitter(img)

        # K_scaled is present in the file, but this model is direct image -> BEV,
        # so we keep it for completeness and debugging only.
        K = sample.get("K_scaled", None)
        if K is None:
            K = np.eye(3, dtype=np.float32)
        K = torch.tensor(K, dtype=torch.float32)

        return {"img": img, "K": K, "mask": mask}


def collate_fn(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    imgs = torch.stack([b["img"] for b in batch], dim=0)
    K = torch.stack([b["K"] for b in batch], dim=0)
    mask = torch.stack([b["mask"] for b in batch], dim=0)
    return {"img": imgs, "K": K, "mask": mask}


# -----------------------------------------------------------------------------
# Losses
# -----------------------------------------------------------------------------

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        probs = probs.flatten(1)
        targets = targets.flatten(1)
        inter = (probs * targets).sum(dim=1)
        den = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2 * inter + self.smooth) / (den + self.smooth)
        return 1.0 - dice.mean()


class BoundaryLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
        sobel_y = torch.tensor([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=torch.float32)
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))

    def _edges(self, x: torch.Tensor) -> torch.Tensor:
        gx = F.conv2d(x, self.sobel_x, padding=1)
        gy = F.conv2d(x, self.sobel_y, padding=1)
        return torch.sqrt(gx * gx + gy * gy + 1e-6)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(logits)
        return F.l1_loss(self._edges(pred), self._edges(targets))


class CompositeLoss(nn.Module):
    def __init__(self, pos_weight: float = 1.0):
        super().__init__()
        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32))
        self.dice = DiceLoss()
        self.boundary = BoundaryLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight)
        dice = self.dice(logits, targets)
        boundary = self.boundary(logits, targets)
        total = 0.55 * bce + 0.30 * dice + 0.15 * boundary
        logs = {"bce": float(bce.item()), "dice": float(dice.item()), "boundary": float(boundary.item())}
        return total, logs


# -----------------------------------------------------------------------------
# Model: ResUNet for image -> BEV
# -----------------------------------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.shortcut = nn.Sequential()
        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.block(x) + self.shortcut(x), inplace=True)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        if x1.shape[-2:] != x2.shape[-2:]:
            x1 = F.interpolate(x1, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class ResUNetBEV(nn.Module):
    def __init__(self, in_channels: int = 3, base_ch: int = 32, out_channels: int = 1):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_ch)
        self.down1 = Down(base_ch, base_ch * 2)
        self.down2 = Down(base_ch * 2, base_ch * 4)
        self.down3 = Down(base_ch * 4, base_ch * 8)
        self.down4 = Down(base_ch * 8, base_ch * 16)

        self.up1 = Up(base_ch * 16, base_ch * 8)
        self.up2 = Up(base_ch * 8, base_ch * 4)
        self.up3 = Up(base_ch * 4, base_ch * 2)
        self.up4 = Up(base_ch * 2, base_ch)
        self.outc = nn.Conv2d(base_ch, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)


# -----------------------------------------------------------------------------
# Metrics
# -----------------------------------------------------------------------------

def binary_iou(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = ((pred + target) > 0).float().sum(dim=(1, 2, 3))
    return (inter + eps) / (union + eps)


def binary_dice(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    den = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (2 * inter + eps) / (den + eps)


def pixel_accuracy(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    correct = (pred == target).float().sum(dim=(1, 2, 3))
    total = torch.numel(target[0])
    return correct / (total + eps)


# -----------------------------------------------------------------------------
# Training / evaluation
# -----------------------------------------------------------------------------

def estimate_pos_weight(ds: Dataset, max_items: int = 50) -> float:
    pos = 0.0
    neg = 0.0
    n = min(len(ds), max_items)
    for i in range(n):
        y = ds[i]["mask"]
        pos += float(y.sum().item())
        neg += float((1.0 - y).sum().item())
    return float(neg / max(pos, 1.0))


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, threshold: float = 0.5) -> Dict[str, float]:
    model.eval()
    all_iou, all_dice, all_acc = [], [], []
    total_time = 0.0
    n_seen = 0

    for batch in loader:
        img = batch["img"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)

        t0 = time.time()
        logits = model(img)
        total_time += time.time() - t0

        probs = torch.sigmoid(logits)
        pred = (probs > threshold).float()

        all_iou.append(binary_iou(pred, mask))
        all_dice.append(binary_dice(pred, mask))
        all_acc.append(pixel_accuracy(pred, mask))
        n_seen += img.shape[0]

    miou = torch.cat(all_iou).mean().item()
    mdice = torch.cat(all_dice).mean().item()
    macc = torch.cat(all_acc).mean().item()
    fps = n_seen / max(total_time, 1e-6)
    return {"mIoU": miou, "Dice": mdice, "PixelAcc": macc, "FPS": fps}


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    loss_fn: CompositeLoss,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler],
    device: torch.device,
    accum_steps: int = 1,
    grad_clip: float = 1.0,
    amp: bool = True,
) -> Dict[str, float]:
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running = {"loss": 0.0, "bce": 0.0, "dice": 0.0, "boundary": 0.0}
    n_batches = len(loader)

    for step, batch in enumerate(loader):
        img = batch["img"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=amp):
            logits = model(img)
            loss, logs = loss_fn(logits, mask)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == n_batches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                try:
                    scheduler.step()
                except ValueError:
                    pass  # Prevent OneCycleLR overflow crash

        running["loss"] += float(loss.item()) * accum_steps
        running["bce"] += logs["bce"]
        running["dice"] += logs["dice"]
        running["boundary"] += logs["boundary"]

    for k in running:
        running[k] /= max(n_batches, 1)
    return running


def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer, scheduler, epoch: int, best_metric: float) -> None:
    torch.save(
        {
            "epoch": epoch,
            "best_metric": best_metric,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
        },
        path,
    )


def load_checkpoint(path: str, model: nn.Module, optimizer=None, scheduler=None, device: str = "cpu") -> Tuple[int, float]:
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt.get("scheduler") is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    return int(ckpt.get("epoch", 0)), float(ckpt.get("best_metric", 0.0))


# -----------------------------------------------------------------------------
# Main entry points
# -----------------------------------------------------------------------------

def build_dataloaders(cfg: CFG):
    train_pkl = cfg.train_pkl or resolve_drive_file("train_metadata.pkl")
    val_pkl = cfg.val_pkl or resolve_drive_file("val_metadata.pkl")

    print("Train PKL:", train_pkl)
    print("Val PKL  :", val_pkl)

    train_ds = SingleViewBEVDataset(
        train_pkl,
        image_size=(cfg.image_h, cfg.image_w),
        mask_size=(cfg.mask_h, cfg.mask_w) if cfg.mask_h and cfg.mask_w else None,
        augment=True,
    )
    val_ds = SingleViewBEVDataset(
        val_pkl,
        image_size=(cfg.image_h, cfg.image_w),
        mask_size=train_ds.mask_size,
        augment=False,
    )

    print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
    print(f"Image size  : {(cfg.image_h, cfg.image_w)}")
    print(f"Mask size   : {train_ds.mask_size}")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
        collate_fn=collate_fn,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
        collate_fn=collate_fn,
        drop_last=False,
    )
    return train_ds, val_ds, train_loader, val_loader


def run_training(cfg: CFG) -> str:
    seed_everything(cfg.seed)
    ensure_dir(cfg.out_dir)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_ds, val_ds, train_loader, val_loader = build_dataloaders(cfg)

    pos_weight = estimate_pos_weight(train_ds, max_items=min(50, len(train_ds)))
    print(f"Estimated BCE pos_weight = {pos_weight:.3f}")

    model = ResUNetBEV(in_channels=3, base_ch=32, out_channels=1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps_per_epoch = math.ceil(len(train_loader) / cfg.accum_steps)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=cfg.lr,
        steps_per_epoch=steps_per_epoch,
        epochs=cfg.epochs,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
    loss_fn = CompositeLoss(pos_weight=pos_weight).to(device)

    best_miou = -1.0
    best_path = os.path.join(cfg.out_dir, cfg.save_name)

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_logs = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            loss_fn=loss_fn,
            scheduler=scheduler,
            device=device,
            accum_steps=cfg.accum_steps,
            grad_clip=cfg.grad_clip,
            amp=cfg.amp,
        )
        val_metrics = evaluate(model, val_loader, device, threshold=cfg.threshold)
        elapsed = time.time() - t0

        print(
            f"Epoch {epoch:02d}/{cfg.epochs} | "
            f"train_loss={train_logs['loss']:.4f} | "
            f"val_mIoU={val_metrics['mIoU']:.4f} | val_Dice={val_metrics['Dice']:.4f} | "
            f"val_Acc={val_metrics['PixelAcc']:.4f} | FPS={val_metrics['FPS']:.2f} | "
            f"{elapsed/60:.1f} min"
        )

        if val_metrics["mIoU"] > best_miou:
            best_miou = val_metrics["mIoU"]
            save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_miou)
            print(f"  -> Saved best checkpoint to {best_path}")

    print(f"Training complete. Best val mIoU: {best_miou:.4f}")
    return best_path


@torch.no_grad()
def run_testing(cfg: CFG, ckpt_path: str, save_predictions_dir: Optional[str] = None) -> Dict[str, float]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, val_ds, _, val_loader = build_dataloaders(cfg)

    model = ResUNetBEV(in_channels=3, base_ch=32, out_channels=1).to(device)
    load_checkpoint(ckpt_path, model, device=device)
    model.eval()

    if save_predictions_dir is not None:
        ensure_dir(save_predictions_dir)

    metrics = evaluate(model, val_loader, device, threshold=cfg.threshold)
    print("Validation metrics:", json.dumps(metrics, indent=2))

    if save_predictions_dir is not None:
        idx = 0
        for batch in val_loader:
            img = batch["img"].to(device, non_blocking=True)
            logits = model(img)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > cfg.threshold).astype(np.uint8)

            for b in range(preds.shape[0]):
                out_path = os.path.join(save_predictions_dir, f"pred_{idx:05d}.npy")
                np.save(out_path, preds[b, 0])
                idx += 1
        print(f"Saved predictions to {save_predictions_dir}")

    return metrics


if __name__ == "__main__":
    cfg = CFG()
    # In Colab, mount Drive first:
    # from google.colab import drive
    # drive.mount('/content/drive')

    # Optional debugging:
    # inspect_pkl(cfg.train_pkl or resolve_drive_file('train_metadata.pkl'), n=1)

    best_ckpt = run_training(cfg)
    run_testing(cfg, best_ckpt, save_predictions_dir=os.path.join(cfg.out_dir, "predictions"))

Device: cuda
Train PKL: /content/drive/MyDrive/Mobility_Hackathon/preprocessed/train_metadata.pkl
Val PKL  : /content/drive/MyDrive/Mobility_Hackathon/preprocessed/val_metadata.pkl
Train samples: 1938 | Val samples: 486
Image size  : (256, 448)
Mask size   : (256, 704)
Estimated BCE pos_weight = 26.765
Epoch 01/25 | train_loss=0.5890 | val_mIoU=0.0570 | val_Dice=0.0939 | val_Acc=0.8900 | FPS=329.00 | 1.2 min
  -> Saved best checkpoint to /content/drive/MyDrive/Mobility Hackathon/checkpoints/best_singleview_bev.pt
Epoch 02/25 | train_loss=0.5077 | val_mIoU=0.0904 | val_Dice=0.1340 | val_Acc=0.9424 | FPS=337.76 | 1.2 min
  -> Saved best checkpoint to /content/drive/MyDrive/Mobility Hackathon/checkpoints/best_singleview_bev.pt
Epoch 03/25 | train_loss=0.4777 | val_mIoU=0.0919 | val_Dice=0.1478 | val_Acc=0.8619 | FPS=315.55 | 1.2 min
  -> Saved best checkpoint to /content/drive/MyDrive/Mobility Hackathon/checkpoints/best_singleview_bev.pt
Epoch 04/25 | train_loss=0.4764 | val_mIoU=0.0696 |